# Module 06 — Workflow: parallelization and evaluator-optimizer

**THE ONE IDEA:** three more of Anthropic's five patterns, and the distinction that gets
probed in interviews — **who decides the subtasks?**

| pattern | shape | who decides the subtasks |
|---|---|---|
| **sectioning** | fan out, aggregate | **you**, in code, ahead of time |
| **voting** | same task N times, take consensus | **you** |
| **evaluator-optimizer** | generate → critique → revise | **you** set the loop |

In all three, *you* did. When the **model** decides them at runtime, the same shape
becomes orchestrator-workers — and that is the boundary where a workflow starts becoming
an agent. Module 30 crosses it.


In [1]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
from _providers import get_client
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import re

client, MODEL, _ = get_client("openai")

def call(prompt, max_tok=400, temp=0.0):
    r = client.chat.completions.create(model=MODEL, max_tokens=max_tok, temperature=temp,
                                       messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip()

print("ready")

ready


## Pattern 1 — Sectioning

Three *independent* subtasks, fanned out in parallel. **I** chose these three sections,
in code, before any model ran. Wall-clock is one call, not three.

In [2]:
import time
SECTIONS = ["the early repayment charge schedule",
            "the maximum loan-to-value rules",
            "the proof-of-income requirements"]

def section(topic):
    return call(f"In 2 sentences, explain {topic} for a UK mortgage.", max_tok=150)

t0 = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:
    parts = list(pool.map(section, SECTIONS))
par = time.time() - t0

for s, p in zip(SECTIONS, parts):
    print(f"\n[{s}]\n  {p[:130]}")
print(f"\n3 calls in {par:.1f}s wall-clock — they ran concurrently, not back to back.")


[the early repayment charge schedule]
  The early repayment charge (ERC) for a UK mortgage is a fee imposed by lenders if you pay off your mortgage, or a portion of it, b

[the maximum loan-to-value rules]
  In the UK, maximum loan-to-value (LTV) ratios for mortgages typically range up to 95%, meaning borrowers can borrow up to 95% of t

[the proof-of-income requirements]
  To apply for a UK mortgage, lenders typically require proof of income such as recent payslips (usually the last three months), ban

3 calls in 2.1s wall-clock — they ran concurrently, not back to back.


## Pattern 2 — Voting

Same question N times, take the consensus. Buys reliability on a task with **one right
answer** and a model that is *nearly* right. It does not fix a model that is confidently
wrong — N wrong answers still agree.

In [3]:
Q = ("A property is valued at 320000 and the loan is 285000. "
     "What is the LTV as a percentage to one decimal place? "
     "Reply with ONLY the number.")

with ThreadPoolExecutor(max_workers=5) as pool:
    votes = list(pool.map(lambda _: call(Q, max_tok=20, temp=1.0), range(5)))

nums = [m.group() for v in votes if (m := re.search(r"\d+\.?\d*", v))]
tally = Counter(nums)
print("votes :", nums)
print("tally :", dict(tally))
print("winner:", tally.most_common(1)[0][0], " (true answer 89.1)")
print("\nUnanimous means low variance, NOT correctness. Module 37 shows why a")
print("bimodal 80% and a uniform 80% are different animals.")

votes : ['89.1', '89.1', '89.1', '89.1', '89.1']
tally : {'89.1': 5}
winner: 89.1  (true answer 89.1)

Unanimous means low variance, NOT correctness. Module 37 shows why a
bimodal 80% and a uniform 80% are different animals.


## Pattern 3 — Evaluator-optimizer

Generate, critique, revise. Needs a **stopping criterion** or it runs forever —
module 28 covers all three (critic says done, output converges, hard cap).

In [4]:
draft = call("Write a 3-sentence rejection letter to a mortgage applicant "
             "declined for insufficient income.", max_tok=200)

for rnd in range(1, 4):
    critique = call(f"Critique this letter for empathy, clarity and regulatory "
                    f"tone. If it is good enough, reply with exactly APPROVED.\n\n{draft}",
                    max_tok=200)
    print(f"\nROUND {rnd} critique: {critique[:110]}")
    if "APPROVED" in critique.upper():
        print("  -> stopping criterion met")
        break
    draft = call(f"Rewrite the letter addressing this critique. Reply with ONLY the\n"
                 f"letter text — no preamble, no closing offer.\n\n"
                 f"CRITIQUE: {critique}\n\nLETTER: {draft}", max_tok=200)
else:
    print("\n  -> hit the hard cap of 3 rounds (the OTHER stopping criterion)")

print("\nFINAL:\n", draft)


ROUND 1 critique: The letter is clear and concise, and it maintains a professional and regulatory-appropriate tone. However, it 

ROUND 2 critique: The letter demonstrates a good balance of empathy and clarity. It acknowledges the applicant's effort, clearly
  -> stopping criterion met

FINAL:
 Dear [Applicant Name],

Thank you for your application for a mortgage with our company. We appreciate the time and effort you invested in the process. After careful review, we regret to inform you that we are unable to approve your loan request at this time due to insufficient income to meet our lending criteria.

We understand this news may be disappointing, and we encourage you to review your financial situation and consider reapplying in the future when your circumstances have improved. If you have any questions or would like guidance on how to strengthen your application, please feel free to contact us.

Sincerely,  
[Your Name]  
[Your Company]


## The lesson

In [5]:
print("LESSON — three patterns, one question: WHO DECIDES THE SUBTASKS?")
print()
print("  sectioning    I listed 3 topics in a Python list         -> workflow")
print("  voting        I chose N=5 and the tie-break rule         -> workflow")
print("  evaluator     I wrote the loop and its stopping rule     -> workflow")
print()
print("Every decision above is in the source. Re-run it and the same calls happen")
print("in the same order. Cost is bounded before you start.")
print()
print("Change ONE thing — let the model decide what the sections should be, per")
print("input — and this becomes ORCHESTRATOR-WORKERS, where cost is unbounded until")
print("you cap it. That single swap is the workflow/agent boundary, and it is the")
print("most common interview probe on this material.")
print()
print("Block B ends here. From module 07 the model takes the wheel.")

LESSON — three patterns, one question: WHO DECIDES THE SUBTASKS?

  sectioning    I listed 3 topics in a Python list         -> workflow
  voting        I chose N=5 and the tie-break rule         -> workflow
  evaluator     I wrote the loop and its stopping rule     -> workflow

Every decision above is in the source. Re-run it and the same calls happen
in the same order. Cost is bounded before you start.

Change ONE thing — let the model decide what the sections should be, per
input — and this becomes ORCHESTRATOR-WORKERS, where cost is unbounded until
you cap it. That single swap is the workflow/agent boundary, and it is the
most common interview probe on this material.

Block B ends here. From module 07 the model takes the wheel.


---

**Next:** `../C_the_loop/07_agent_without_sdk_react.ipynb` — the model starts choosing.